# #5 Staging plots

## Purpose

Plot distributions metrics comparing embryos across stages and to published references from Pijuan-Sala et al. 2019 and Qiu et al. 2024.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from devmap.config import embryos, set_theme, stage_palette
from devmap.utils import get_paths, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("staging")

## Pijuan-Sala et al. 2019

In [ ]:
adata = sc.read_h5ad("../data/pijuan-sala19_integrated_scvi.h5ad")
sc.pp.neighbors(adata, use_rep='X_scvi')
sc.tl.umap(adata, random_state=0)

Dataset

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=600)
sc.pl.umap(adata, color="dataset", ax = ax)
save_plot(plots_path / "pijuan-sala19_integrated_umap_dataset.svg", fig)

Stage

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=600)
sc.pl.umap(adata, color="stage", ax = ax, palette=stage_palette)
save_plot(plots_path / "pijuan-sala19_integrated_umap_stage.svg", fig)

Transcriptional similarity

In [ ]:
df = pd.read_csv(results_path / "embryo_cross_early_sinkhorn_matrix.csv", index_col=0)
fig = plt.figure(figsize=(3,3), dpi=200)
sns.heatmap(
    df,
    cmap="magma_r",
    cbar_kws={'label': 'Sinkhorn distance'},
    square=True,
)
save_plot("integration/pijuan-sala19_embryo_cross_sinkhorn_heatmap.svg", fig)

## Qiu et al. 2024

In [ ]:
adata = sc.read_h5ad("../data/qiu24_integrated_scvi.h5ad")
sc.pp.neighbors(adata, use_rep='X_scvi')
sc.tl.umap(adata, random_state=0)

Dataset

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=600)
sc.pl.umap(adata, color="dataset", ax = ax)
save_plot(plots_path / "qiu24_integrated_umap_dataset.svg", fig)

Stage

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=600)
sc.pl.umap(adata, color="stage", ax = ax, palette=stage_palette)
save_plot(plots_path / "qiu24_integrated_umap_stage.svg", fig)

Transcrptional similarity

In [ ]:
df = pd.read_csv(results_path / "embryo_cross_late_sinkhorn_matrix.csv", index_col=0)
fig = plt.figure(figsize=(3,3), dpi=200)
sns.heatmap(
    df,
    cmap="magma_r",
    cbar_kws={'label': 'Sinkhorn distance'},
    square=True,
)
save_plot("integration/qiu24_embryo_cross_sinkhorn_heatmap.svg", fig)

## Pairwise transcriptional similarity

In [ ]:
def to_symmetric_matrix(df_pw, value_col, order):
    """Aggregate batches to mean, then fill both triangles."""
    agg = df_pw.groupby(["embryo_a", "embryo_b"])[value_col].mean()
    idx = {e: k for k, e in enumerate(order)}
    mat = pd.DataFrame(np.nan, index=order, columns=order)
    for (a, b), v in agg.items():
        mat.iloc[idx[a], idx[b]] = v
        mat.iloc[idx[b], idx[a]] = v
    np.fill_diagonal(mat.values, 0.0)
    return mat

In [ ]:
df = pd.read_csv(results_path / "pairwise_embryo_distributional_distances.csv")
mat = to_symmetric_matrix(df, "sinkhorn", np.unique(df[["embryo_a", "embryo_b"]].values.flatten()))
mat = mat.loc[embryos, embryos]
mat.values[np.diag_indices_from(mat)] = np.nan
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=600) 
sns.heatmap(mat, xticklabels=mat.columns, yticklabels=mat.index, cmap="magma_r", ax = ax, square = True)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
save_plot(plots_path / "pairwise_embryo_distances.svg", rasterize = True)

## 